# Final Exam, Spring 2024: Topic identification
_Version 1.0a-c148515_

*All of the header information is important. Please read it.*

**Topics number of exercises:** This problem builds on your knowledge of basic Python, pandas, Numpy, clustering, and matrix factorization. It has **9** exercises numbered 0 to **8**. There are **16** available points. However to earn 100% the threshold is **12** points. (Therefore once you hit **12** points you can stop. There is no extra credit for exceeding this threshold.)

**Exercise ordering:** Each exercise builds logically on previous exercises but you may solve them in any order. That is if you can't solve an exercise you can still move on and try the next one. Use this to your advantage as the exercises are **not** necessarily ordered in terms of difficulty. Higher point values generally indicate more difficult exercises. 

**Demo cells:** Code cells starting with the comment `### define demo inputs` load results from prior exercises applied to the entire data set and use those to build demo inputs. These must be run for subsequent demos to work properly but they do not affect the test cells. The data loaded in these cells may be rather large (at least in terms of human readability). You are free to print or otherwise use Python to explore them but we may not print them in the starter code.

**Debugging you code:** Right before each exercise test cell there is a block of text explaining the variables available to you for debugging. You may use these to test your code and can print/display them as needed (careful when printing large objects you may want to print the head or chunks of rows at a time).

**Exercise points breakdown:**


- Ex. 0 - `get_record`: **2** point(s)

- Ex. 1 - `clean_record`: **1** point(s)

- Ex. 2 - `records_to_metadf`: **1** point(s)

- Ex. 3 - `gen_corpus`: **2** point(s)

- Ex. 4 - `build_cluster_matrix`: **2** point(s)

- Ex. 5 - `get_top_tokens`: **2** point(s)

- Ex. 6 - `find_distinctive_tokens`: **3** point(s)

- Ex. 7 - `lsa_svd_cluster__FREE`: **1** point(s)

- Ex. 8 - `get_nmf_clusters`: **2** point(s)


**Final reminders:** 

- Submit after **every exercise**
- Review the generated grade report after you submit to see what errors were returned
- Stay calm, skip problems as needed and take short breaks at your leisure

# Environment setup #

Run the following cells to set up the problem.

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import scipy as sp
import pandas as pd

import dill
from pprint import pprint

# The problem: Mining a computer-document corpus for "topics" #

**Your overall task.** In this problem, the dataset is the metadata for a collection of computer science research papers and technical books. Your task will be to try to identify **topics**, or collections of similar documents, using three different algorithms:

- Algorithm 1: K-means.
- Algorithm 2: An SVD-based method.
- Algorithm 3: NMF – nonnegative matrix factorization.

You won't have to implement these algorithms, but you will have to prepare the data and figure _how_ to run code that implements them.

**The dataset.** The dataset comes from the [AMiner collection](https://www.aminer.org/citation). Run the following cell to load the raw data into a variable named `raw_records` and display a sample.

In [2]:
with open('resource/asnlib/publicdata/acm-v9-sample-100k.dill', 'rb') as fp:
    raw_records = dill.load(fp)

print(f"=== Success: Loaded {len(raw_records):,} raw records. ===")
print(f"\nExample: Records 0, 7, and 15:\n")
pprint([raw_records[k] for k in [0, 7, 15]])

=== Success: Loaded 100,000 raw records. ===

Example: Records 0, 7, and 15:

[['#*Computationally efficient algorithms for a one-time pad scheme\n',
  '#@A. G. Akritas, S. S. Lyengar, A. A. Rampuria\n',
  '#t1984\n',
  '#cInternational Journal of Parallel Programming\n',
  '#index6\n'],
 ['#*Extended person-machine interface\n',
  '#@Rachel Reichman-Adar\n',
  '#t1984\n',
  '#cArtificial Intelligence\n',
  '#index108\n'],
 ['#*Algorithms for on-the-fly garbage collection\n',
  '#@Mordechai Ben-Ari\n',
  '#t1984\n',
  '#cACM Transactions on Programming Languages and Systems (TOPLAS)\n',
  '#index269\n',
  '#%317992\n',
  '#%320265\n',
  '#%320491\n',
  '#%598024\n',
  '#%2135000\n']]


The `raw_records` data structure is a list of **raw records**. Each raw record represents the metadata for a technical paper or book, stored as a list of raw (unprocessed) strings. Each string is a **raw attribute**.

Each raw attribute encodes some meta-information about the paper, like its title, list of authors, year of publication, and so on. The attribute starts with a short prefix beginning with `#`. The prefix indicates what the attribute is, and it is immediately followed by its string value, with **no delimiters** separating the prefix from the value.

Here are the possible prefixes and their meaning:

| Prefix   | Meaning |
|----------|---------|
| `#*`     | Title of the work |
| `#@`     | Author names |
| `#t`     | Year of publication |
| `#c`     | Where the work was published |
| `#index` | ID number of this work |
| `#%`     | ID number of a work that this work cites |
| `#!`     | Abstract (short description of the work) |

Compare these prefixes to the example raw records printed above and observe the following:
1. **Not all prefixes are present** for a given raw record.
2. The `#%` prefix represents a citation, that is, the ID number of a work that this work cites. A raw record may have **more than one** `#%` attribute. 

**Stopwords.** Per common practice, our text analysis will involve **ignoring** certain words, referred to as **stopwords**. The code cell below will load a pre-selected set of such words for some of the demos that follow.

In [3]:
with open('resource/asnlib/publicdata/STOPWORDS.dill', 'rb') as fp:
    STOPWORDS = dill.load(fp)

print("A sample of stopwords:")
sorted(list(STOPWORDS))[:7]

A sample of stopwords:


['a', 'about', 'above', 'acm', 'after', 'again', 'against']

# Ex. 0 (2pts): `get_record` #

**Your task:** Define `get_record` as follows:

Convert a single 'raw' record into a specially formatted Python dictionary.

**Input**: `lines`: A list of strings, where each string is a raw attribute.

**Return**: `record`: A Python dictionary, constructed as follows.

**Requirements/steps**: For each raw attribute, convert its prefix into a human-readable key (see below) and extract its value as a string.
Recall that a raw attribute, like `'#%'`, may appear more than once.
Therefore, the output record should associate each key with a **sorted list** of all values encountered.

The human-readable key corresponding to each raw attribute is:
- `'#*'`: `'title'`
- `'#@'`: `'authors'`
- `'#t'`: `'year'`
- `'#c'`: `'venue'`
- `'#index'`: `'id'`
- `'#%'`: `'refs'`
- `'#!'`: `'abstract'`

**Other notes**: If a raw attribute appears only once, it should _still_ appear in a list. (The list will be one-element long in this case.)


**Example**. A correct implementation should produce, for the demo, the following output:
```python
{'authors': ['Mordechai Ben-Ari'],
 'id': ['269'],
 'refs': ['2135000', '317992', '320265', '320491', '598024'],
 'title': ['Algorithms for on-the-fly garbage collection'],
 'venue': ['ACM Transactions on Programming Languages and Systems (TOPLAS)'],
 'year': ['1984']}
```


In [26]:
### Solution - Exercise 0  

def get_record(lines: list) -> dict:
    from collections import defaultdict
    
    final_dict = defaultdict(list)
    
    for line in lines:
        if line[0:6] == '#index':
            final_dict['id'].append(line[6:].strip())
        elif line[0:2] == '#*':
            final_dict['title'].append(line[2:].strip())
        elif line[0:2] == '#@':
            final_dict['authors'].append(line[2:].strip())
        elif line[0:2] == '#t':
            final_dict['year'].append(line[2:].strip())
        elif line[0:2] == '#c':
            final_dict['venue'].append(line[2:].strip())
        elif line[0:2] == '#%':
            final_dict['refs'].append(line[2:].strip())
        elif line[0:2] == '#!':
            final_dict['abstract'].append(line[2:].strip())
            
    for x in final_dict:
        final_dict[x].sort()
                    
    return final_dict
    
### Demo function call

pprint(get_record(raw_records[15]))

defaultdict(<class 'list'>,
            {'authors': ['Mordechai Ben-Ari'],
             'id': ['269'],
             'refs': ['2135000', '317992', '320265', '320491', '598024'],
             'title': ['Algorithms for on-the-fly garbage collection'],
             'venue': ['ACM Transactions on Programming Languages and Systems '
                       '(TOPLAS)'],
             'year': ['1984']})


 <!-- Test Cell Boilerplate -->  
**Test cell.** The cell below will test your solution for get_record (exercise 0). The testing variables will be available for debugging under the following names in a dictionary format.  
- `input_vars` - Input variables for your solution.   
- `original_input_vars` - Copy of input variables from prior to running your solution. Any `key:value` pair in `original_input_vars` should also exist in `input_vars` - otherwise the inputs were modified by your solution.  
- `returned_output_vars` - Outputs returned by your solution.  
- `true_output_vars` - The expected output. This _should_ "match" `returned_output_vars` based on the question requirements - otherwise, your solution is not returning the correct output. 

In [321]:
### Test Cell - Exercise 0  

from cse6040_devkit.tester_fw.testers import Tester
from yaml import safe_load

with open('resource/asnlib/publicdata/assignment_config.yaml') as f:
    ex_conf = safe_load(f)['exercises']['get_record']['config']

ex_conf['func'] = get_record

tester = Tester(ex_conf, key=b'4oTuD3jYU_dfQvrxIzSmKJQBDeiWMaODi2nmk0sQk1o=', path='resource/asnlib/publicdata/')
for _ in range(75):
    try:
        tester.run_test()
        (input_vars, original_input_vars, returned_output_vars, true_output_vars) = tester.get_test_vars()
    except:
        (input_vars, original_input_vars, returned_output_vars, true_output_vars) = tester.get_test_vars()
        raise

###
### AUTOGRADER TEST - DO NOT REMOVE
###

print('Passed! Please submit.')

Passed! Please submit.


## Run me! ##

Whether your solution is working or not, run the following code cell, which will preload a list of processed records (dictionaries) in the global variable, `records`.

In [28]:
with open('resource/asnlib/publicdata/records.dill', 'rb') as fp:
    records = dill.load(fp)

print(f"Sample post-processed records:")
for k in [0, 7, 15]:
    print(f"\n=== get_record(raw_records[{k}]) ===")
    pprint(records[k])

Sample post-processed records:

=== get_record(raw_records[0]) ===
{'authors': ['A. G. Akritas, S. S. Lyengar, A. A. Rampuria'],
 'id': ['6'],
 'title': ['Computationally efficient algorithms for a one-time pad scheme'],
 'venue': ['International Journal of Parallel Programming'],
 'year': ['1984']}

=== get_record(raw_records[7]) ===
{'authors': ['Rachel Reichman-Adar'],
 'id': ['108'],
 'title': ['Extended person-machine interface'],
 'venue': ['Artificial Intelligence'],
 'year': ['1984']}

=== get_record(raw_records[15]) ===
{'authors': ['Mordechai Ben-Ari'],
 'id': ['269'],
 'refs': ['2135000', '317992', '320265', '320491', '598024'],
 'title': ['Algorithms for on-the-fly garbage collection'],
 'venue': ['ACM Transactions on Programming Languages and Systems (TOPLAS)'],
 'year': ['1984']}


# Ex. 1 (1pts): `clean_record` #

**Your task:** Define `clean_record` as follows:

Clean a (non-raw) dictionary record.

**Input**: `record` — A Python dictionary as might be produced by `get_record`.

**Return**: `cleaned` — A copy of `record` cleaned as explained below, or `None` if any required keys are missing.

**Requirements/steps**: The goal of this task is two-fold: 1) Ensure that the record has certain "mandatory" attributes, and 2) simplify the values for certain single-value attributes. More specifically, your function should satisfy these requirements.

- If the input is missing _any_ of the following keys, **return `None`**: `['id', 'title', 'year', 'venue', 'abstract']`.
- For the `'id'` and `'year'` keys, assume there is only one value in the list and convert it into an integer and store this value.
- If there is a `'refs'` key, convert each element of its list to an integer and store this list of integers.
- For any other keys that are present in `record`, assume there is only one value and store that value as-is (i.e., as a string).

Return the cleaned dictionary (or `None` per the first requirement).


**Example**. A correct implementation should produce, for the demo, the following output:
```python
None
{'abstract': 'A mathematical model for communicating sequential processes '
             'isgiven, and a number of its interesting and useful properties '
             'arestated and proved. The possibilities of nondetermimsm are '
             'fullytaken into account.',
 'authors': 'S. D. Brookes, C. A. R. Hoare, A. W. Roscoe',
 'id': 319,
 'refs': [2135000, 318212, 320203, 374129, 408527, 547420, 555361, 565837,
          566544, 566549, 680046, 689430],
 'title': 'A Theory of Communicating Sequential Processes',
 'venue': 'Journal of the ACM (JACM)',
 'year': 1984}
```


In [32]:
records[0]

{'title': ['Computationally efficient algorithms for a one-time pad scheme'],
 'authors': ['A. G. Akritas, S. S. Lyengar, A. A. Rampuria'],
 'year': ['1984'],
 'venue': ['International Journal of Parallel Programming'],
 'id': ['6']}

In [47]:
### Solution - Exercise 1  

def clean_record(record: dict) -> dict:
    from collections import defaultdict
    
    final_dict = defaultdict(list)
    
    try:
        final_dict['id'] = int(record['id'][0])
        final_dict['title'] = record['title'][0]
        final_dict['year'] = int(record['year'][0])
        final_dict['venue'] = record['venue'][0]
        final_dict['abstract'] = record['abstract'][0]
    
    except:
        return None
    
    try:
        final_dict['authors'] = record['authors'][0]
        for element in record['refs']:
            final_dict['refs'].append(int(element))
    except:
        pass
        
    return final_dict

### Demo function call

pprint(clean_record(records[0])) # None!
pprint(clean_record(records[17])) # Valid

None
defaultdict(<class 'list'>,
            {'abstract': 'A mathematical model for communicating sequential '
                         'processes isgiven, and a number of its interesting '
                         'and useful properties arestated and proved. The '
                         'possibilities of nondetermimsm are fullytaken into '
                         'account.',
             'authors': 'S. D. Brookes, C. A. R. Hoare, A. W. Roscoe',
             'id': 319,
             'refs': [2135000,
                      318212,
                      320203,
                      374129,
                      408527,
                      547420,
                      555361,
                      565837,
                      566544,
                      566549,
                      680046,
                      689430],
             'title': 'A Theory of Communicating Sequential Processes',
             'venue': 'Journal of the ACM (JACM)',
             'year': 1984})


 <!-- Test Cell Boilerplate -->  
**Test cell.** The cell below will test your solution for clean_record (exercise 1). The testing variables will be available for debugging under the following names in a dictionary format.  
- `input_vars` - Input variables for your solution.   
- `original_input_vars` - Copy of input variables from prior to running your solution. Any `key:value` pair in `original_input_vars` should also exist in `input_vars` - otherwise the inputs were modified by your solution.  
- `returned_output_vars` - Outputs returned by your solution.  
- `true_output_vars` - The expected output. This _should_ "match" `returned_output_vars` based on the question requirements - otherwise, your solution is not returning the correct output. 

In [48]:
### Test Cell - Exercise 1  

from cse6040_devkit.tester_fw.testers import Tester
from yaml import safe_load

with open('resource/asnlib/publicdata/assignment_config.yaml') as f:
    ex_conf = safe_load(f)['exercises']['clean_record']['config']

ex_conf['func'] = clean_record

tester = Tester(ex_conf, key=b'BhZT_oWZZqOjWxKNFXA5A7uL44XtRE6X7vsh1IcmmOs=', path='resource/asnlib/publicdata/')
for _ in range(75):
    try:
        tester.run_test()
        (input_vars, original_input_vars, returned_output_vars, true_output_vars) = tester.get_test_vars()
    except:
        (input_vars, original_input_vars, returned_output_vars, true_output_vars) = tester.get_test_vars()
        raise

###
### AUTOGRADER TEST - DO NOT REMOVE
###

print('Passed! Please submit.')

Passed! Please submit.


## Run me! ##

Whether your solution is working or not, run the following code cell, which will preload a list of cleaned records in the global variable, `cleaned_records`.

In [49]:
with open('resource/asnlib/publicdata/papers.dill', 'rb') as fp:
    cleaned_records = dill.load(fp)

print("Examples of cleaned records:")
for k in [0, len(cleaned_records)//2, len(cleaned_records)-1]:
    record = cleaned_records[k]
    print(f"\n=== Record {k} ===")
    pprint(record)

Examples of cleaned records:

=== Record 0 ===
{'abstract': 'A mathematical model for communicating sequential processes '
             'isgiven, and a number of its interesting and useful properties '
             'arestated and proved. The possibilities of nondetermimsm are '
             'fullytaken into account.',
 'authors': 'S. D. Brookes, C. A. R. Hoare, A. W. Roscoe',
 'id': 319,
 'refs': [2135000,
          318212,
          320203,
          374129,
          408527,
          547420,
          555361,
          565837,
          566544,
          566549,
          680046,
          689430],
 'title': 'A Theory of Communicating Sequential Processes',
 'venue': 'Journal of the ACM (JACM)',
 'year': 1984}

=== Record 36141 ===
{'abstract': 'Despite the dawn of the XML era, semantic interoperability '
             'issues still remain unsolved. As various initiatives trying to '
             'address how the underlying business information should be '
             'modelled, nam

# Ex. 2 (1pts): `records_to_metadf` #

**Your task:** Define `records_to_metadf` as follows:

Convert (cleaned) records, which represent publications metadata, into a pandas `DataFrame`.

**Input**: `records` – A list of _cleaned_ records (e.g., see `clean_records`).

**Return**: `metadf` – A pandas `DataFrame` of the "metadata, as defined below."

**Requirements/steps**: Construct a pandas `DataFrame` with columns for these keys:
`'id'`, `'title'`, `'year'`, `'venue'`, `'abstract'`

**Other notes**: The target fields are guaranteed to exist based on the cleaning procedure.


**Example**. The demo runs the function on elements [11222, 11239, 12329] of `cleaned_records`. A correct implementation would produce:

|     id | title                   |   year | venue                   | abstract                |
|-------:|:------------------------|-------:|:------------------------|:------------------------|
| 648046 | A Tutorial for CC++     |   1994 | A Tutorial for CC++     | No abstract available.  |
| 648461 | Logics of Programs      |   1989 | Logics of Programs      | None Available          |
| 673426 | Codes and Number Theory |   1997 | Codes and Number Theory | Codes and Number Theory |


In [54]:
### Solution - Exercise 2  

def records_to_metadf(records: list) -> pd.DataFrame:
    df = pd.DataFrame(records)
    df = df[['id', 'title', 'year', 'venue', 'abstract']]
    return df

### Demo function call

demo_cleaned_records = [cleaned_records[k] for k in [11222, 11239, 12329]]
demo_metadf = records_to_metadf(demo_cleaned_records)
display(demo_metadf)

,id,title,year,venue,abstract
0,648046,A Tutorial for CC++,1994,A Tutorial for CC++,No abstract available.
1,648461,Logics of Programs,1989,Logics of Programs,None Available
2,673426,Codes and Number Theory,1997,Codes and Number Theory,Codes and Number Theory


 <!-- Test Cell Boilerplate -->  
**Test cell.** The cell below will test your solution for records_to_metadf (exercise 2). The testing variables will be available for debugging under the following names in a dictionary format.  
- `input_vars` - Input variables for your solution.   
- `original_input_vars` - Copy of input variables from prior to running your solution. Any `key:value` pair in `original_input_vars` should also exist in `input_vars` - otherwise the inputs were modified by your solution.  
- `returned_output_vars` - Outputs returned by your solution.  
- `true_output_vars` - The expected output. This _should_ "match" `returned_output_vars` based on the question requirements - otherwise, your solution is not returning the correct output. 

In [55]:
### Test Cell - Exercise 2  

from cse6040_devkit.tester_fw.testers import Tester
from yaml import safe_load

with open('resource/asnlib/publicdata/assignment_config.yaml') as f:
    ex_conf = safe_load(f)['exercises']['records_to_metadf']['config']

ex_conf['func'] = records_to_metadf

tester = Tester(ex_conf, key=b'hgW1Cvvo9Kg4sTImuGlhMic09mEdkjjQQFPMCiw5gaE=', path='resource/asnlib/publicdata/')
for _ in range(75):
    try:
        tester.run_test()
        (input_vars, original_input_vars, returned_output_vars, true_output_vars) = tester.get_test_vars()
    except:
        (input_vars, original_input_vars, returned_output_vars, true_output_vars) = tester.get_test_vars()
        raise

###
### AUTOGRADER TEST - DO NOT REMOVE
###

print('Passed! Please submit.')

Passed! Please submit.


## Run me! ##

Whether your solution is working or not, run the following code cell, which will preload a dataframe of metadata records in the global variable, `metadf`.

In [56]:
with open('resource/asnlib/publicdata/metadf.dill', 'rb') as fp:
    metadf = dill.load(fp)

# For Ex. 3, break dependence with a working Ex. 2
demo_metadf = metadf.loc[[11222, 11239, 12329]].reset_index(drop=True)

print("Examples of metadata records:")
display(metadf.sample(5))

Examples of metadata records:


,id,title,year,venue,abstract
15783,793767,Task models and interaction models in a multip...,2004,TAMODIA '04 Proceedings of the 3rd annual conf...,"In a Multiple User Interfaces (MUI) context, s..."
66783,1886867,Towards the Design of Systolic Genetic Search,2012,IPDPSW '12 Proceedings of the 2012 IEEE 26th I...,"This paper elaborates on a new, fresh parallel..."
20410,895105,A neural network classification of credit appl...,2006,AIA'06 Proceedings of the 24th IASTED internat...,The paper aims to find an efficient model for ...
20534,897738,Image Translation and Rotation on Hexagonal St...,2006,CIT '06 Proceedings of the Sixth IEEE Internat...,Image translation and rotation are becoming es...
54655,1636934,Research on the Hydrodynamic of the SKUD 18 Cl...,2011,ICFCSE '11 Proceedings of the 2011 Internation...,The paper is using the MAXSURF software to mod...


# Ex. 3 (2pts): `gen_corpus` #

**Your task:** Define `gen_corpus` as follows:

Construct a "document corpus" that can be fed into some text analysis tools. This corpus consists of "pseudo-documents," built up from the documents' "tokens."

**Inputs**:
* `metadf` – A dataframe with document metadata, as produced by `records_to_metadf`.
* `stopwords` - A Python set containing tokens to _ignore_.

**Return**: `pseudodf` – A new dataframe where tokens have been extracted and filtered from some of the attributes, formatted as explained below.

**Requirements/steps**: Your dataframe should have the following columns:
- `'id'`: The document ID, taken as-is from the input, `metadf['id']`.
- `'title'`: The document title, taken as-is from `metadf['title']`.
- `'pseudodoc'`: A "pseudo-document," which is a string built from the `'title'`, `'venue'`, and `'abstract'` as follows.

Recall that each row of `metadf` represents a document.
For each document, its _pseudo-document_ is constructed by extracting _tokens_ from lowercase versions of its title, venue, and abstract strings.
When extracting tokens, regard a consecutive sequence of plain alphabetic characters, `'a'` through `'z'`, as a token. (Regex may be helpful here!)
To construct the pseudo-document from these tokens, first sort the **unique** tokens, remove any **stopwords**, and then join the remaining tokens using a single space character, `' '`, to separate them.

**Other notes**: Per the above requirements, do not forget to:
1. extract your unique tokens from lowercase versions of title, venue, and abstract;
2. remove any stopwords using the input `stopwords` (and **not** the global variable, `STOPWORDS`, which is for demos); and
3. concatenate only the unique tokens, in sorted order, and separated by spaces.


**Example**: Suppose we run `gen_corpus` on the demo metadata dataframe from the previous exercise.
A correct implementation would return:

|     id | title                   | pseudodoc                      |
|-------:|:------------------------|:-------------------------------|
| 648046 | A Tutorial for CC++     | abstract available cc tutorial |
| 648461 | Logics of Programs      | available logics none programs |
| 673426 | Codes and Number Theory | codes number theory            |


In [111]:
### Solution - Exercise 3  

def gen_corpus(metadf, stopwords):
    import re
    from collections import defaultdict

    df = metadf.copy()
    
    df['combined_str'] = df['title'].str.lower() + ' ' + df['venue'].str.lower() + ' ' + df['abstract'].str.lower()
    
    data_dict = defaultdict(list)
    for index, row in df.iterrows():
        data_dict['id'].append(row['id'])
        token = re.findall('([a-z]+)', row['combined_str'])
        token = set(token)
        token = [t for t in token if t not in stopwords]
        token.sort()
        data_dict['pseudodoc'].append(' '.join(token))
        
    tokens_df = pd.DataFrame.from_dict(data_dict)
    
    df = df.merge(tokens_df, on='id')
    
    return df[['id', 'title', 'pseudodoc']]

### Demo function call
demo_corpus = gen_corpus(demo_metadf, STOPWORDS)
display(demo_corpus)

,id,title,pseudodoc
0,648046,A Tutorial for CC++,abstract available cc tutorial
1,648461,Logics of Programs,available logics none programs
2,673426,Codes and Number Theory,codes number theory


 <!-- Test Cell Boilerplate -->  
**Test cell.** The cell below will test your solution for gen_corpus (exercise 3). The testing variables will be available for debugging under the following names in a dictionary format.  
- `input_vars` - Input variables for your solution.   
- `original_input_vars` - Copy of input variables from prior to running your solution. Any `key:value` pair in `original_input_vars` should also exist in `input_vars` - otherwise the inputs were modified by your solution.  
- `returned_output_vars` - Outputs returned by your solution.  
- `true_output_vars` - The expected output. This _should_ "match" `returned_output_vars` based on the question requirements - otherwise, your solution is not returning the correct output. 

In [108]:
returned_output_vars['gen_corpus_output']

,id,pseudodoc
0,1493957,advantages an and are artificial be binary cal...
1,1629756,advantage allow analyses and applied architect...
2,1731647,advection analysis analytic and applications a...
3,1353367,according algorithm an analysis and applied ap...


In [109]:
true_output_vars['gen_corpus_output']

,id,title,pseudodoc
47856,1493957,Policy gradients for cryptanalysis,advantages an and are artificial be binary cal...
54320,1629756,A highly reliable gpu-based raid system,advantage allow analyses and applied architect...
59220,1731647,New numerical methods for the Riesz space frac...,advection analysis analytic and applications a...
41227,1353367,Parallel ODE-solvers with stepsize control,according algorithm an analysis and applied ap...


In [115]:
### Test Cell - Exercise 3  

from cse6040_devkit.tester_fw.testers import Tester
from yaml import safe_load

with open('resource/asnlib/publicdata/assignment_config.yaml') as f:
    ex_conf = safe_load(f)['exercises']['gen_corpus']['config']

ex_conf['func'] = gen_corpus

tester = Tester(ex_conf, key=b'o0kicBPd2HinEbuciVMQHNjfCLJj4eVAWPBqKCfSeNw=', path='resource/asnlib/publicdata/')
for _ in range(75):
    try:
        tester.run_test()
        (input_vars, original_input_vars, returned_output_vars, true_output_vars) = tester.get_test_vars()
    except:
        (input_vars, original_input_vars, returned_output_vars, true_output_vars) = tester.get_test_vars()
        raise

###
### AUTOGRADER TEST - DO NOT REMOVE
###

print('Passed! Please submit.')

Passed! Please submit.


## Run me! ##

Whether your solution is working or not, run the following code cell, which will preload a corpus of pseudo-documents in the global dataframe, `corpusdf`.

In [116]:
with open('resource/asnlib/publicdata/corpus.dill', 'rb') as fp:
    corpusdf = dill.load(fp)

print(f"A sample of the pseudo-document corpus:")
display(corpusdf.sample(4))

A sample of the pseudo-document corpus:


,id,title,pseudodoc
19909,884566,Detection of Scale Intervals in Digital Images,accurate achieve appears calculate cyclic data...
68348,1920505,A study of terminology auditors' performance f...,acceptable according accuracy achieve assigned...
3527,376134,Going Digital: A Musician's Guide to Technology,able afraid book charts computers covers diagr...
52120,1584263,CytoModeler,access analysis application availability avail...


# Mini-tutorial (read & run me!): k-means clustering + matrix factorization #

A "classical" way to represent the corpus for data mining is in the bag of words model. In this model, we transform a corpus consisting of $n$ documents and $m$ unique tokens into an $n \times m$ (sparse) matrix $X$, where element $x_{ij}$ "measures" the strength of the relationship between document $i$ and token $j$.

Using one of scikit-learn's [text feature extraction tools](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html#), we constructed a matrix from the corpus. Run the next code cell to load the matrix into the global variable, `X`.

In [117]:
with open('resource/asnlib/publicdata/X.dill', 'rb') as fp:
    X = dill.load(fp)
    
print(f"The shape of `X`: {X.shape}")
print(f"- Smallest value: {X.min()}")
print(f"- Largest value: {X.max()}")
print(f"- Mean value: {X.mean()}")

The shape of `X`: (72282, 108145)
- Smallest value: 0.0
- Largest value: 1.0
- Mean value: 6.840847576497508e-05


Here is "spy plot" of `X`, which visualizes the locations of the nonzero entries.

![Spy plot showing the nonzero structure of the sparse matrix, `X`](resource/asnlib/publicdata/spy-X.png)

**A topic model via k-means.** Given `X`, we can then use scikit-learn's K-means implementation to cluster the documents. It works by (magically) calculating a vector-space representation of the documents of `X` and then running K-means on it similar to what you did in Notebook 14. We can interpret each cluster as a **"topic"** that groups documents containing similar tokens together.

To save a little time, we ran this clustering for you to try to find 10 topics (clusters). _(The code we used appears at the end of the exam.)_

Run the cell below to load the precomputed clusters, which consists of the following:
- `km_sizes`: The number of documents in each cluster.
- `km_labels`: A 1-D array where `km_labels[i]` is the the cluster ID number (0-9) to which `i` is assigned.
- `km_centers`: A 2-D array where row `km_centers[j, :]` is the coordinate of the center of cluster `j`.

In [118]:
with open('resource/asnlib/publicdata/kmeans.dill', 'rb') as fp:
    _, km_sizes, km_centers, km_labels = dill.load(fp)

print(f"- Cluster sizes:", km_sizes)
print(f"- Cluster labels:", km_labels)
print(f"- Cluster centers, {km_centers.shape[0]:,} x {km_centers.shape[1]}, where each column is a centroid:")
pprint(km_centers)

- Cluster sizes: [10136  6764  4193  5724  5270  2855  7475  9599 13564  6702]
- Cluster labels: [7 7 7 ... 6 5 6]
- Cluster centers, 108,145 x 10, where each column is a centroid:
array([[1.60634653e-05, 1.10854239e-04, 0.00000000e+00, ...,
        6.51034007e-05, 2.60202046e-05, 0.00000000e+00],
       [0.00000000e+00, 2.31752030e-04, 1.76940759e-04, ...,
        0.00000000e+00, 5.29350135e-05, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
        0.00000000e+00, 5.24392544e-05, 1.89033683e-05],
       ...,
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
        2.26129576e-05, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 2.06486921e-05, 0.00000000e+00, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00]])


**The "matrix factorization" interpretation of k-means.** Although we commonly think of k-means as clustering points, we can also view what it does as computing a certain kind of **matrix factorization.**

Recall that $X$ had $m$ rows (documents) and $n$ columns (tokens), and we identified $k$ topics using k-means clustering. Let $C$ be the $n \times k$ matrix of centroids that resulted.

What k-means does is assign each document $i$ to one of these centroids. Let $R$ be a $m \times k$ (sparse) matrix where entry $r_{is} = 1$ if document $i$ is assigned to topic $s$ and equals 0 otherwise. Then one way to interpret k-means is to say that it **approximates** the matrix $X$ by the product,

$$X \approx R \cdot C^T.$$

To see why, consider let $\hat{r}_i^T$ be the $i$-th **row** of $R$ and $c_s$ be the $s$-th **column** of $C$. By the definition of matrix product,

$$\hat{r}_i^T \cdot C^T
= \hat{r_i}^T \Biggl[\begin{matrix} \\ c_0 & \cdots & c_s & \cdots & c_{k-1} \\ \\ \end{matrix}\Biggl]^T
= \left[\begin{matrix} r_{i0} & \cdots & r_{is} & \cdots & r_{i,k-1} \end{matrix}\right] \cdot \left[\begin{matrix} c_0^T \\ \vdots \\ c_s^T \\ \vdots \\ c_{k-1}^T \end{matrix}\right]
= \sum_{s} r_{is} c_s^T.
$$

However, since k-means assigned document $i$ to exactly one cluster $s$, then only one term in the sum is nonzero, which is $c_s$, the centroid for cluster $s$. In other words, the action of $R \cdot C^T$ is to assign each document (row) of $X$ to one centroid.

With this concept in mind, you are now ready for the next exercise.

# Ex. 4 (2pts): `build_cluster_matrix` #

**Your task:** Define `build_cluster_matrix` as follows:

Given a k-means clustering result, construct a k-means assignment matrix. From the preceding discussion, it is the $R$ matrix.

**Inputs**:
- `labels`: A 1-D array where `labels[i]` is the cluster ID assigned to document `i`.
- `max_label`: The largest cluster ID plus 1. That is, `0 <= labels[i] < max_label` for all `i`.

**Return**: `R`, a Scipy COO matrix ([scipy.sparse.coo_matrix](https://docs.scipy.org/doc/scipy/reference/generated/scipy.sparse.coo_matrix.html)).

**Requirements/steps:** The returned matrix `R` should have `len(labels)` rows and `max_label` columns.
It should only have entries equal to 1.0 at positions `(i, j)` where `j == labels[i]`;
all other entries should be regarded as zero values and, therefore, not stored explicitly.    


**Example**: Suppose we run:
```python
  build_cluster_matrix(np.array([1, 2, 0, 0, 2, 2, 2, 3, 0, 3]), 4)
```

A correct implementation would return:
```python
    (0, 1)	1.0
  (1, 2)	1.0
  (2, 0)	1.0
  (3, 0)	1.0
  (4, 2)	1.0
  (5, 2)	1.0
  (6, 2)	1.0
  (7, 3)	1.0
  (8, 0)	1.0
  (9, 3)	1.0
```

Here is a picture (spy plot):
![Spy plot: demo `R` matrix](resource/asnlib/publicdata/demo_R.png)


In [137]:
### Solution - Exercise 4  

def build_cluster_matrix(labels: np.ndarray, max_label: int) -> sp.sparse.coo_matrix:
    import numpy as np
    from scipy.sparse import coo_matrix
    
    data = []
    row = []
    col = []
    for i in range(len(labels)):
        data.append(1.0)
        row.append(i)
        col.append(labels[i])
        
    return coo_matrix((data, (row, col)), shape=(len(labels), max_label), dtype=np.int8)

### Demo function call
demo_R = build_cluster_matrix(np.array([1, 2, 0, 0, 2, 2, 2, 3, 0, 3]), 4)
print(demo_R)

  (0, 1)	1
  (1, 2)	1
  (2, 0)	1
  (3, 0)	1
  (4, 2)	1
  (5, 2)	1
  (6, 2)	1
  (7, 3)	1
  (8, 0)	1
  (9, 3)	1


 <!-- Test Cell Boilerplate -->  
**Test cell.** The cell below will test your solution for build_cluster_matrix (exercise 4). The testing variables will be available for debugging under the following names in a dictionary format.  
- `input_vars` - Input variables for your solution.   
- `original_input_vars` - Copy of input variables from prior to running your solution. Any `key:value` pair in `original_input_vars` should also exist in `input_vars` - otherwise the inputs were modified by your solution.  
- `returned_output_vars` - Outputs returned by your solution.  
- `true_output_vars` - The expected output. This _should_ "match" `returned_output_vars` based on the question requirements - otherwise, your solution is not returning the correct output. 

In [138]:
### Test Cell - Exercise 4  

from cse6040_devkit.tester_fw.testers import Tester
from yaml import safe_load

with open('resource/asnlib/publicdata/assignment_config.yaml') as f:
    ex_conf = safe_load(f)['exercises']['build_cluster_matrix']['config']

ex_conf['func'] = build_cluster_matrix

tester = Tester(ex_conf, key=b'cub7HYh_bdtYjSXeD5UPThVliqosSf9XWu4LLxZjlOs=', path='resource/asnlib/publicdata/')
for _ in range(75):
    try:
        tester.run_test()
        (input_vars, original_input_vars, returned_output_vars, true_output_vars) = tester.get_test_vars()
    except:
        (input_vars, original_input_vars, returned_output_vars, true_output_vars) = tester.get_test_vars()
        raise

###
### AUTOGRADER TEST - DO NOT REMOVE
###

print('Passed! Please submit.')

Passed! Please submit.


## Run me! ##

Whether your solution is working or not, run the following code cell, which will preload the k-means cluster matrix, $R$, for $X$ (`X`), in the global variable `R_km`.

In [139]:
with open('resource/asnlib/publicdata/R.dill', 'rb') as fp:
    R_km = dill.load(fp)
    
R_km

<72282x10 sparse matrix of type '<class 'numpy.float64'>'
	with 72282 stored elements in COOrdinate format>

# Ex. 5 (2pts): `get_top_tokens` #

**Your task:** Define `get_top_tokens` as follows:

Given a cluster, identify its most frequent tokens.

**Inputs**:
- `cid`: The ID of a cluster to analyze.
- `labels`: Cluster assignments, where document `i` was assigned to cluster `labels[i]`.
- `corpusdf`: A corpus / pseudo-document dataframe (see Ex. 3, `gen_corpus`), having the columns `'id'`, `'title'`, and `'pseudodoc'`.
- `k`: The number of tokens to return.

**Return**: `top_tokens`, a Python set of the `k` most frequent tokens.

**Requirements/steps**:
1. Use `labels` to identify the documents belonging to cluster `cid`.
2. Select their corresponding pseudo-documents `corpusdf`. See the note below.
3. For each unique token, count the number of pseudo-documents in which it appeared.
4. Return the `k` most frequently occurring tokens as a Python set.
In the case of ties, consider tokens in ascending order.

**Other notes**:
- To match a document ID `i` to its pseudo-document, note that `i` is the _index_ value of `corpusdf`
(and, in particular, **not** the `'id'` column!).
- If there are fewer than `k` unique tokens, return as many as available.
- If there are ties, use the token itselfsort the tokens by name.


**Example**: For the demo code, a correct implementation should return:
```python
  {'page', 'reviews', 'academic', 'book'}
```


In [162]:
### Solution - Exercise 5  

def get_top_tokens(cid: int, labels: np.ndarray, corpusdf: pd.DataFrame, k=10) -> set:
    import re
    from collections import defaultdict
    
    id_list = [i for i, label in enumerate(labels) if label == cid]
    
    df = corpusdf[corpusdf.index.isin(id_list)]
    
    token_list = []
    for idx, row in df.iterrows():
        token = re.findall('([a-z]+)', row['pseudodoc'])
        for t in token:
            token_list.append(t)
    
    token_dict = defaultdict(int)
    for token in token_list:
        token_dict[token] += 1
        
    token_dict = dict(sorted(token_dict.items(), key=lambda x:(-x[1], x[0])))
    
    token_set = set()
    for i, token in enumerate(token_dict.keys()):
        if i < k:
            token_set.add(token)
    
    return token_set
        
        
### Demo function call
with open('resource/asnlib/publicdata/demo_args_get_top_tokens.dill', 'rb') as fp:
    demo_cid, demo_labels, demo_corpusdf, demo_k = dill.load(fp)
print(get_top_tokens(demo_cid, demo_labels, demo_corpusdf, demo_k))

{'book', 'reviews', 'page', 'academic'}


 <!-- Test Cell Boilerplate -->  
**Test cell.** The cell below will test your solution for get_top_tokens (exercise 5). The testing variables will be available for debugging under the following names in a dictionary format.  
- `input_vars` - Input variables for your solution.   
- `original_input_vars` - Copy of input variables from prior to running your solution. Any `key:value` pair in `original_input_vars` should also exist in `input_vars` - otherwise the inputs were modified by your solution.  
- `returned_output_vars` - Outputs returned by your solution.  
- `true_output_vars` - The expected output. This _should_ "match" `returned_output_vars` based on the question requirements - otherwise, your solution is not returning the correct output. 

In [163]:
### Test Cell - Exercise 5  

from cse6040_devkit.tester_fw.testers import Tester
from yaml import safe_load

with open('resource/asnlib/publicdata/assignment_config.yaml') as f:
    ex_conf = safe_load(f)['exercises']['get_top_tokens']['config']

ex_conf['func'] = get_top_tokens

tester = Tester(ex_conf, key=b'BBpXHA04_pc-q0tx10aYqwXtRUfiJeKp90vRPTLzC5k=', path='resource/asnlib/publicdata/')
for _ in range(75):
    try:
        tester.run_test()
        (input_vars, original_input_vars, returned_output_vars, true_output_vars) = tester.get_test_vars()
    except:
        (input_vars, original_input_vars, returned_output_vars, true_output_vars) = tester.get_test_vars()
        raise

###
### AUTOGRADER TEST - DO NOT REMOVE
###

print('Passed! Please submit.')

Passed! Please submit.


## Run me! ##

We ran one of our reference solutions for `get_top_tokens` on `corpusdf` to extract top tokens for all the clusters discovered by a K-means run. Run the following code cell to load them into a global dictionary called `top_tokens_km`.

In [164]:
with open('resource/asnlib/publicdata/top_tokens.dill', 'rb') as fp:
    km_top_tokens = dill.load(fp)
    
km_top_tokens

{0: {'analysis',
  'control',
  'data',
  'model',
  'performance',
  'simulation',
  'system',
  'systems',
  'time',
  'two'},
 1: {'applications',
  'computing',
  'data',
  'distributed',
  'information',
  'network',
  'service',
  'services',
  'system',
  'systems'},
 2: {'mobile',
  'network',
  'networks',
  'nodes',
  'performance',
  'show',
  'simulation',
  'systems',
  'time',
  'wireless'},
 3: {'applications',
  'data',
  'design',
  'high',
  'parallel',
  'performance',
  'show',
  'system',
  'systems',
  'time'},
 4: {'algorithm',
  'analysis',
  'data',
  'image',
  'images',
  'information',
  'processing',
  'recognition',
  'show',
  'two'},
 5: {'algorithm',
  'artificial',
  'data',
  'information',
  'intelligence',
  'learning',
  'model',
  'system',
  'systems',
  'volume'},
 6: {'algorithm',
  'analysis',
  'data',
  'information',
  'methods',
  'model',
  'performance',
  'show',
  'systems',
  'two'},
 7: {'algorithm',
  'algorithms',
  'linear',
  'nu

# Ex. 6 (3pts): `find_distinctive_tokens` #

**Your task:** Define `find_distinctive_tokens` as follows:

Suppose you are given the top tokens of several clusters.
For each cluster, determine its "most distinctive" tokens and keep only those.

**Inputs**:
- `token_sets`: A dictionary, where `token_sets[i]` are the top tokens appearing in cluster `i`.
- `max_occur`: The maximum number of allowable occurrences of a token across clusters for that token to be considered "distinctive" of that cluster.

**Return**:
- `distinctive`: A new dictionary mapping cluster `key` to a (possibly smaller) set of tokens `distictive[key]`.

**Requirements/steps**:
1. For each token, determine how many clusters it belongs to.
2. For each cluster, consider its _distinctive tokens_ to be only those that appear in _at most_ `max_occur` clusters. If a cluster has no distinctive tokens, do **not** include it in the output.
3. Construct and return a new dictionary mapping each cluster ID to its distinctive tokens.


**Example**: For the demo code, a correct implementation should return:
```python
{0: {'analysis', 'simulation', 'control'}, 4: {'processing', 'images', 'image', 'analysis', 'recognition'}, 6: {'analysis', 'methods'}, 2: {'network', 'nodes', 'networks', 'simulation', 'wireless', 'mobile'}, 1: {'network', 'distributed', 'service', 'computing', 'services', 'applications'}, 3: {'parallel', 'applications', 'design', 'high'}, 9: {'engineering', 'programming', 'language', 'software', 'applications', 'design', 'development'}, 8: {'work', 'technology', 'study', 'research', 'design', 'development'}, 5: {'artificial', 'intelligence', 'learning', 'volume'}, 7: {'problems', 'algorithms', 'number', 'linear', 'theory', 'set'}}
```


In [171]:
### Solution - Exercise 6  

def find_distinctive_tokens(token_sets: dict, max_occur=3) -> dict:
    from collections import defaultdict
    
    my_dict = defaultdict(int)
    
    for key, value in token_sets.items():
        for token in value:
            my_dict[token] += 1
        
    distinct_dict = {k: v for k, v in my_dict.items() if v <= max_occur}
    
    distinctive = defaultdict(set)
    for key, value in token_sets.items():
        for token in value:
            if token in distinct_dict.keys():
                distinctive[key].add(token)
    
    return distinctive

### Demo function call
print(find_distinctive_tokens(km_top_tokens))

defaultdict(<class 'set'>, {0: {'simulation', 'analysis', 'control'}, 1: {'network', 'distributed', 'computing', 'service', 'services', 'applications'}, 2: {'networks', 'network', 'mobile', 'wireless', 'simulation', 'nodes'}, 3: {'parallel', 'applications', 'high', 'design'}, 4: {'recognition', 'image', 'analysis', 'images', 'processing'}, 5: {'artificial', 'intelligence', 'learning', 'volume'}, 6: {'methods', 'analysis'}, 7: {'number', 'problems', 'set', 'theory', 'algorithms', 'linear'}, 8: {'design', 'study', 'development', 'research', 'work', 'technology'}, 9: {'programming', 'language', 'engineering', 'design', 'development', 'software', 'applications'}})


 <!-- Test Cell Boilerplate -->  
**Test cell.** The cell below will test your solution for find_distinctive_tokens (exercise 6). The testing variables will be available for debugging under the following names in a dictionary format.  
- `input_vars` - Input variables for your solution.   
- `original_input_vars` - Copy of input variables from prior to running your solution. Any `key:value` pair in `original_input_vars` should also exist in `input_vars` - otherwise the inputs were modified by your solution.  
- `returned_output_vars` - Outputs returned by your solution.  
- `true_output_vars` - The expected output. This _should_ "match" `returned_output_vars` based on the question requirements - otherwise, your solution is not returning the correct output. 

In [172]:
### Test Cell - Exercise 6  

from cse6040_devkit.tester_fw.testers import Tester
from yaml import safe_load

with open('resource/asnlib/publicdata/assignment_config.yaml') as f:
    ex_conf = safe_load(f)['exercises']['find_distinctive_tokens']['config']

ex_conf['func'] = find_distinctive_tokens

tester = Tester(ex_conf, key=b'5VeeLsiYTK9AUwJ8sN5jiO4xU3sWzBJooV29Q6R7Erk=', path='resource/asnlib/publicdata/')
for _ in range(75):
    try:
        tester.run_test()
        (input_vars, original_input_vars, returned_output_vars, true_output_vars) = tester.get_test_vars()
    except:
        (input_vars, original_input_vars, returned_output_vars, true_output_vars) = tester.get_test_vars()
        raise

###
### AUTOGRADER TEST - DO NOT REMOVE
###

print('Passed! Please submit.')

Passed! Please submit.


# Ex. 7 (1pts): `lsa_svd_cluster__FREE` #

**Your task:** Define `lsa_svd_cluster__FREE` as follows:

**FREE EXERCISE**: Use the SVD and K-means to compute a clustering.
This method is sometimes referred to as _latent semantic analysis_ (LSA).

**Background**: In the data-matrix factorization view of K-means, we compute $X \sim R C^T$ and use $R$ to identify clusters.
That suggests we could try _other_ factorization methods, including the singular value decomposition (SVD) from Notebook 15.

In this strategy, we compute an $d$-truncated SVD, $X \approx U_d \Sigma_d V_d^T$, where we choose $d$ to be "small" compared to the number of columns of $X$.
In so doing, the matrix $Y \equiv U_d \Sigma_d$ acts as lower-dimensional representation of $X$, where each row of $Y$ is a $d$-dimensional version of the corresponding row of $X$.
We can then run K-means on $Y$.

Here, we are giving you some code to carry out this analysis.
This code uses scikit-learn's [TruncatedSVD](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.TruncatedSVD.html#) and [K-means](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html#) implementations.
Inspect the results and then move on to the next (and last) part and exercise.

**Inputs**:
- `X`: An `n`-by-`m` data matrix where rows are data points.
- `num_clusters`: The desired number of clusters to identify.
- `num_components`: The truncated dimension, $d$.

**Return:** `labels`, an array of cluster labels.
The `i`-th row of `X` is "assigned" to a cluster whose ID is `labels[i]`.
These labels will lie in the range 0 to `num_clusters-1`.


In [173]:
### Solution - Exercise 7  

def lsa_svd_cluster__FREE(X: np.ndarray, num_clusters=10, num_components=10):
    """**FREE EXERCISE**: Use the SVD and K-means to compute a clustering.
This method is sometimes referred to as _latent semantic analysis_ (LSA).

**Background**: In the data-matrix factorization view of K-means, we compute $X \sim R C^T$ and use $R$ to identify clusters.
That suggests we could try _other_ factorization methods, including the singular value decomposition (SVD) from Notebook 15.

In this strategy, we compute an $d$-truncated SVD, $X \\approx U_d \Sigma_d V_d^T$, where we choose $d$ to be "small" compared to the number of columns of $X$.
In so doing, the matrix $Y \equiv U_d \Sigma_d$ acts as lower-dimensional representation of $X$, where each row of $Y$ is a $d$-dimensional version of the corresponding row of $X$.
We can then run K-means on $Y$.

Here, we are giving you some code to carry out this analysis.
This code uses scikit-learn's [TruncatedSVD](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.TruncatedSVD.html#) and [K-means](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html#) implementations.
Inspect the results and then move on to the next (and last) part and exercise.

**Inputs**:
- `X`: An `n`-by-`m` data matrix where rows are data points.
- `num_clusters`: The desired number of clusters to identify.
- `num_components`: The truncated dimension, $d$.

**Return:** `labels`, an array of cluster labels.
The `i`-th row of `X` is "assigned" to a cluster whose ID is `labels[i]`.
These labels will lie in the range 0 to `num_clusters-1`.
    """
    # Step 1: Compute the (truncated) SVD of X: X ~ U S V^T
    from sklearn.decomposition import TruncatedSVD
    svd = TruncatedSVD(n_components=num_components, n_iter=20, random_state=1_234)
    Y = svd.fit_transform(X)
    
    # Step 2: Run K-means on U
    from sklearn.cluster import KMeans
    kmeans = KMeans(n_clusters=num_clusters, max_iter=100, n_init=1, random_state=5_678).fit(Y)
    labels = kmeans.predict(Y)
    return labels

### Demo function call
pass # bug? this line gets omitted in the output
with open('resource/asnlib/publicdata/X.dill', 'rb') as fp:
    X = dill.load(fp)
    print(f"Data matrix (`X`) shape: {X.shape[0]:,} x {X.shape[1]:,}")
lsa_labels = lsa_svd_cluster__FREE(X, num_clusters=10)
print(f"Cluster assignments: {lsa_labels}")

print(f"\nDistinctive top tokens for LSA:")
with open('resource/asnlib/publicdata/lsa_tokens.dill', 'rb') as fp:
    lsa_top_tokens, lsa_distinctive = dill.load(fp)
pprint(lsa_distinctive)

Data matrix (`X`) shape: 72,282 x 108,145
Cluster assignments: [8 4 8 ... 2 2 2]

Distinctive top tokens for LSA:
{0: {'processing', 'recognition', 'images', 'image'},
 1: {'present', 'language', 'programming', 'software'},
 2: {'learning', 'algorithms'},
 3: {'distributed', 'user', 'computing', 'service', 'services'},
 4: {'parallel', 'high'},
 5: {'study', 'development', 'technology', 'research'},
 6: {'networks', 'network', 'mobile', 'wireless', 'simulation'},
 7: {'simulation', 'control', 'signal'},
 8: {'problems', 'number', 'set', 'theory', 'algorithms', 'linear'},
 9: {'publisher', 'web', 'book', 'guide', 'software', 'students', 'technology'}}


 <!-- Test Cell Boilerplate -->  
**Test cell.** The cell below will test your solution for lsa_svd_cluster__FREE (exercise 7). The testing variables will be available for debugging under the following names in a dictionary format.  
- `input_vars` - Input variables for your solution.   
- `original_input_vars` - Copy of input variables from prior to running your solution. Any `key:value` pair in `original_input_vars` should also exist in `input_vars` - otherwise the inputs were modified by your solution.  
- `returned_output_vars` - Outputs returned by your solution.  
- `true_output_vars` - The expected output. This _should_ "match" `returned_output_vars` based on the question requirements - otherwise, your solution is not returning the correct output. 

In [174]:
### Test Cell - Exercise 7  


print('Passed! Please submit.')

Passed! Please submit.


# Nonnegative Matrix Factorization #

For the final exercise, we will try one more clustering method: nonnegative matrix factorization, or NMF.

Recall that the SVD method starts by reducing the dimension of an $n \times m$ data-matrix $X$ to a $n \times d$ matrix $Y$, where $d \ll m$, via the SVD. Although the qualitative results in Exercise 7 appear reasonable, the matrix $Y$ is not easy to interpret because its entries assume arbitrary values. By contrast, the K-means assignment matrix $R$ is directly related to cluster assignments (although the centroids themselves are arbirary).

In an NMF, we again approximate $X$ by the product of two matrices,
$$X \approx W \cdot H^T,$$
where for some choice of $d < n$, $W$ is $n \times d$ and $H^T$ is $d \times m$ and, importantly, we constrain both $W$ and $H$ to have **nonnegative** entries, meaning greater than or equal to 0.

Each row of $W$ is now a lower dimensional representation of its corresponding data point (row) in $X$ like $U_d \Sigma_d$ in the SVD, but because of nonnegativity, its entries can be viewed as _weights_. That is, $w_{ij}$ measures the strength of association between data point $i$ and column $j$. That in turn implies that if we wish to use $W$ for clustering, we can assign $i$ to the logical cluster $j$ where $w_{ij}$ is highest.

**Computing the NMF.** The following function calculates the NMF for a given input matrix $X$:

```python
def calc_nmf_W(X, num_clusters=10):
    from sklearn.decomposition import NMF
    model = NMF(n_components=num_clusters, init='random', random_state=8_917)
    W = model.fit_transform(X)
    return W
```

We used this function to pre-calculate a `W` for `X`. The code cell below loads this `W`:

In [175]:
with open('resource/asnlib/publicdata/W.dill', 'rb') as fp:
    W = dill.load(fp)
    
print(W.shape)

(72282, 10)


In the final exercise, you'll extract clusters from a given `W`.

# Ex. 8 (2pts): `get_nmf_clusters` #

**Your task:** Define `get_nmf_clusters` as follows:

Determine cluster assignments from an NMF-generated $W$ matrix.

**Input**: `W`, an `n`-by-`m` matrix produced by an NMF computation.

**Return**:
- `labels`: Cluster assignments: for each row `i` of `W`, `labels[i]` is the cluster (column) assignment.
- `ids`: Cluster IDs: a Numpy array containing the values from 0 to `m`-1.
- `sizes`: Cluster sizes: `sizes[c]` is the number of rows of `W` that are assigned to cluster `c`.

**Requirements/steps**:
1. Let `m` be the number of clusters, which is equal to the number of columns of `W`.
2. For each row `i` of `W`, assign it to the cluster `c` having the largest value of `W[i,c]`.
3. The `ids` array is simply a Numpy array of length `m` whose values go from 0 to `m`-1.
4. For each cluster `c`, count how many rows were assigned to it and store that as `sizes[c]`.


**Example**: For the demo code, a correct implementation should return:
```python
(array([1, 0, 2, 0, 2, 1, 3, 0, 0]), array([0, 1, 2, 3]), array([4, 2, 2, 1]))
```


In [204]:
### Solution - Exercise 8  

def get_nmf_clusters(W: np.ndarray) -> (np.ndarray, np.ndarray, np.ndarray):
    import numpy as np
    
    labels = []
    ids = []
    for row in W:
        labels.append(row.argmax())
        
    for i in range(len(W[0])):
        ids.append(i)
                
    sizes = [0] * len(ids)
    for c in ids:
        for label in labels:
            if label == c:
                sizes[c] += 1
                
    
    
    return (np.array(labels), np.array(ids), np.array(sizes))

### Demo function call
with open('resource/asnlib/publicdata/demo_get_nmf_clusters_W.dill', 'rb') as fp:
    demo_W = dill.load(fp)
print("Demo input W:")
pprint(demo_W)
print("\nYour output:")
print(get_nmf_clusters(demo_W))

Demo input W:
array([[0.00000000e+00, 3.37360731e-03, 5.59753284e-05, 0.00000000e+00],
       [7.70853381e-03, 2.13894249e-03, 0.00000000e+00, 2.50804597e-03],
       [0.00000000e+00, 1.11113220e-04, 5.72929128e-03, 0.00000000e+00],
       [1.58384215e-02, 0.00000000e+00, 2.45565407e-03, 2.93780779e-03],
       [0.00000000e+00, 1.67490852e-03, 6.83444761e-03, 3.54015172e-03],
       [2.62927826e-04, 3.43856570e-03, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 1.41361041e-03, 1.39216635e-03, 3.37744335e-03],
       [3.12936562e-03, 0.00000000e+00, 2.07367238e-03, 2.32231424e-03],
       [2.51065173e-03, 3.12451401e-04, 1.15266037e-03, 0.00000000e+00]])

Your output:
(array([1, 0, 2, 0, 2, 1, 3, 0, 0]), array([0, 1, 2, 3]), array([4, 2, 2, 1]))


 <!-- Test Cell Boilerplate -->  
**Test cell.** The cell below will test your solution for get_nmf_clusters (exercise 8). The testing variables will be available for debugging under the following names in a dictionary format.  
- `input_vars` - Input variables for your solution.   
- `original_input_vars` - Copy of input variables from prior to running your solution. Any `key:value` pair in `original_input_vars` should also exist in `input_vars` - otherwise the inputs were modified by your solution.  
- `returned_output_vars` - Outputs returned by your solution.  
- `true_output_vars` - The expected output. This _should_ "match" `returned_output_vars` based on the question requirements - otherwise, your solution is not returning the correct output. 

In [205]:
### Test Cell - Exercise 8  

from cse6040_devkit.tester_fw.testers import Tester
from yaml import safe_load

with open('resource/asnlib/publicdata/assignment_config.yaml') as f:
    ex_conf = safe_load(f)['exercises']['get_nmf_clusters']['config']

ex_conf['func'] = get_nmf_clusters

tester = Tester(ex_conf, key=b'SGeoIKLnOAbhAQCj3ymEM2J0nfjbk6CylhmBcXTw3s4=', path='resource/asnlib/publicdata/')
for _ in range(75):
    try:
        tester.run_test()
        (input_vars, original_input_vars, returned_output_vars, true_output_vars) = tester.get_test_vars()
    except:
        (input_vars, original_input_vars, returned_output_vars, true_output_vars) = tester.get_test_vars()
        raise

###
### AUTOGRADER TEST - DO NOT REMOVE
###

print('Passed! Please submit.')

Passed! Please submit.


# _Fin_ #

If you have made it this far, congratulations on completing it.

**Don't forget to submit!**

**Postscript.** If you are curious, the NMF method produces these distinctive tokens:

```
{0: {'computing', 'high', 'applications', 'distributed', 'parallel'},
 1: {'mobile', 'networks', 'applications', 'network', 'wireless'},
 2: {'analysis', 'computational', 'equations', 'model', 'numerical',
     'problems', 'solution'},
 3: {'processing', 'signal', 'error', 'noise'},
 4: {'technology', 'user', 'development', 'research', 'study', 'design'},
 5: {'learning', 'analysis', 'show', 'model'},
 6: {'images', 'analysis', 'recognition', 'show', 'image'},
 7: {'present', 'theory', 'known', 'set', 'number', 'algorithms', 'show'},
 8: {'development', 'language', 'applications', 'software', 'design', 'model'},
 9: {'power', 'technology', 'high', 'control', 'simulation', 'design'}}
 ```